# Timed FSMs

A timed FSM has transitions that depend on elapsed time, such as timeouts. `DoorAlarm` uses the `sismic` library: a door that stays open for 30 seconds goes into `alarm`. Time comes from an injected `SimulatedClock`, so we control it and the runs are deterministic (no sleeping).

```mermaid
stateDiagram-v2
    [*] --> closed
    closed --> open: open
    open --> closed: close
    open --> alarm: after(30s)
    alarm --> closed: close
```

The timer measures time spent in the current state, so leaving `open` cancels it and re-entering restarts it.

In [1]:
from petrilab.fsm.timed import TIMEOUT_S, DoorAlarm

d = DoorAlarm()
print("timeout:", TIMEOUT_S, "s; initial state:", d.state)
d.send("open")
d.advance(29)
d.state

timeout: 30 s; initial state: closed


'open'

After 29 seconds the door is still `open`: the timeout has not yet elapsed.

In [2]:
d.advance(1)
d.state

'alarm'

At exactly 30 seconds the `after(30)` guard becomes true and the machine moves to `alarm`.

## Closing cancels the timer

In [3]:
d = DoorAlarm()
d.send("open")
d.advance(20)
d.send("close")
d.advance(100)
d.state

'closed'

The door was closed after 20 seconds, so no alarm is raised however long we wait afterwards.

## Re-opening restarts the timer

In [4]:
d = DoorAlarm()
d.send("open")
d.advance(20)
d.send("close")
d.send("open")
d.advance(20)
first = d.state
d.advance(10)
first, d.state

('open', 'alarm')

Total open time is 40 s but split in two periods of 20 s, so the first check is still `open`. After a further 10 s of the second period the alarm fires.

## Closing from alarm and unknown events

In [5]:
d = DoorAlarm()
d.send("open")
d.advance(31)
print(d.state)
d.send("close")
print(d.state)
try:
    d.send("kick")
except ValueError as exc:
    print("ValueError:", exc)

alarm
closed
ValueError: unknown event 'kick'


An alarmed door can be closed to return to `closed`. Events other than `open`/`close` are rejected by the wrapper with `ValueError`.